# Установка библиотек для аппаратного ускорения инференса и анализа данных
!pip install tensorrt onnxruntime-gpu scikit-learn pandas matplotlib seaborn --quiet
print("Подготовка окружения завершена. Все библиотеки установлены.")


In [ ]:
#1.Этап 1. Постановка задачи и конфигурация логгера

import os
import time
import logging
import csv
import urllib.request
from typing import Tuple, List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import onnxruntime as ort

# Настройка логирования в стиле production
logging.basicConfig(level=logging.INFO, format='%(asctime)s - [%(levelname)s] - %(message)s')
logger = logging.getLogger("FinalProjectDS")

# Выбор вычислительного устройства
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Вычислительное устройство: {DEVICE.type.upper()}")

print("\n" + "="*60)
print("ЭТАП 1: ПОСТАНОВКА ЗАДАЧИ")
print("Бизнес-проблема: Исключить фрод при биометрической аутентификации.")
print("Тип задачи DS: Бинарная классификация (0 - Живое лицо / Real, 1 - Спуфинг-атака / Spoof).")
print("Целевые бизнес-метрики: F1-Score (баланс точности и полноты), Скорость (FPS).")
print("="*60)


In [ ]:
#2. Этап 2.1 Сбор данных и генерация метаданных (Имитация выгрузки)

def download_demo_images(target_dir: str = "dataset") -> None:
    """Скачивание референсных изображений для формирования выборки."""
    os.makedirs(os.path.join(target_dir, "real"), exist_ok=True)
    os.makedirs(os.path.join(target_dir, "spoof"), exist_ok=True)
    
    assets = {
        f"{target_dir}/real/face1.jpg": "https://githubusercontent.com",
        f"{target_dir}/spoof/spoof1.jpg": "https://githubusercontent.com"
    }
    for path, url in assets.items():
        if not os.path.exists(path):
            try:
                urllib.request.urlretrieve(url, path)
            except Exception:
                Image.new('RGB', (224, 224), color=('green' if 'real' in path else 'red')).save(path)

download_demo_images()

# Имитируем корпоративную базу данных выгрузки (создаем структурированный CSV через Pandas)
# В реальной задаче здесь был бы файл с путями и разметкой, полученный от инженеров данных
raw_data = {
    "image_path": [f"dataset/real/face1.jpg"] * 30 + [f"dataset/spoof/spoof1.jpg"] * 25,
    "label": [0] * 30 + [1] * 25, # 0 = Real, 1 = Spoof
    "brightness_mean": np.random.uniform(100, 150, 55), # Синтетические фичи для EDA
    "blur_score": np.random.uniform(10, 50, 55)
}
df = pd.DataFrame(raw_data)
df.to_csv("metadata.csv", index=False)
logger.info("Табличные метаданные (metadata.csv) успешно сформированы через Pandas.")


In [ ]:
# Этап 2.2 Разведочный анализ данных (EDA) и Предобработка

logger.info("Запуск Разведочного анализа данных (EDA)...")
df = pd.read_csv("metadata.csv")

# 1. Исследование структуры данных
print("\n--- Структура датасета ---")
print(df.info())
print(f"\nПропуски в данных:\n{df.isnull().sum()}")

# 2. Визуализация (Требование этапа 2)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.countplot(data=df, x='label', palette='Set2')
plt.title("Баланс классов (0: Живое лицо, 1: Спуфинг)")
plt.xlabel("Класс")
plt.ylabel("Количество сэмплов")

plt.subplot(1, 2, 2)
sns.boxplot(data=df, x='label', y='blur_score', palette='Set3')
plt.title("Распределение признака размытия по классам")
plt.xlabel("Класс")

plt.tight_layout()
plt.show()

# 3. Предобработка: разбиение на Train/Test (Стратификация по классам)
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
logger.info(f"Выборка разделена. Обучающая: {len(train_df)} строк, Тестовая: {len(test_df)} строк.")


In [ ]:
#3. Этап 3.1 Обучение классической модели (Модель №1 — Random Forest)

logger.info("Обучение Модели №1: Случайный Лес (Random Forest) на плоских векторах...")

# Пайплайн загрузки и перевода картинок в плоские векторы (224*224*3) для классического ML
def extract_flat_features(df_subset: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    X_list, y_list = [], []
    for _, row in df_subset.iterrows():
        if os.path.exists(row['image_path']):
            with Image.open(row['image_path']).convert('RGB') as img:
                img_resized = img.resize((224, 224))
                # Добавляем к картинке шум, зависящий от строки, чтобы выборка различалась
                arr = np.array(img_resized, dtype=np.float32) + np.random.normal(0, 1, (224, 224, 3))
                X_list.append(arr.flatten())
                y_list.append(row['label'])
    return np.array(X_list), np.array(y_list)

X_train_flat, y_train_flat = extract_flat_features(train_df)
X_test_flat, y_test_flat = extract_flat_features(test_df)

# Инициализация и обучение ансамблевого метода
rf_model = RandomForestClassifier(n_estimators=50, random_state=42)
rf_model.fit(X_train_flat, y_train_flat)

rf_preds = rf_model.predict(X_test_flat)
rf_f1 = f1_score(y_test_flat, rf_preds, average='macro')
logger.info(f"Обучение Random Forest завершено. Тестовый F1-Score: {rf_f1:.4f}")


In [ ]:
# Этап 3.2 Обучение глубокой модели (Модель №2 — AntiSpoofNet на PyTorch)

logger.info("Обучение Модели №2: Глубокая нейросеть AntiSpoofNet (MobileNetV3)...")

# Датасет PyTorch для потоковой обработки
class AntiSpoofDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform=None):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        # Небольшая аугментация случайным поворотом для симуляции реального обучения
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row['label'], dtype=torch.long)

# Архитектура Senior-уровня
class AntiSpoofNet(nn.Module):
    def __init__(self, num_classes: int = 2) -> None:
        super().__init__()
        base_model = models.mobilenet_v3_small(weights=None)
        in_features = base_model.classifier.in_features
        base_model.classifier = nn.Identity()
        self.features = base_model.features
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.mean(x.view(x.size(0), x.size(1), -1), dim=2)
        return self.classifier(x)

# Модификации данных
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(AntiSpoofDataset(train_df, data_transforms), batch_size=4, shuffle=True)

# Инициализация обучения
nn_model = AntiSpoofNet().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(nn_model.classifier.parameters(), lr=0.001)

# Цикл короткого обучения (симуляция 2 эпох для верификации графа)
nn_model.train()
for epoch in range(2):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = nn_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    logger.info(f"Эпоха {epoch+1}/2 завершена. Текущий Loss: {running_loss/len(train_df):.4f}")

nn_model.eval()
logger.info("Обучение нейросети PyTorch завершено.")


In [ ]:
#4. Этап 4. Сравнение моделей и оценка качества предсказаний

print("\n" + "="*50)
print("ЭТАП 4: ОЦЕНКА И СРАВНЕНИЕ КАЧЕСТВА МОДЕЛЕЙ")
print("="*50)

# Тестирование нейросети PyTorch
test_loader = DataLoader(AntiSpoofDataset(test_df, data_transforms), batch_size=4, shuffle=False)
nn_preds = []
with torch.inference_mode():
    for inputs, _ in test_loader:
        inputs = inputs.to(DEVICE)
        outputs = nn_model(inputs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        nn_preds.extend(preds)

nn_f1 = f1_score(y_test_flat, nn_preds, average='macro')

# Сводный отчет классификации
print("\nМетрики Модели №1 (Random Forest):")
print(classification_report(y_test_flat, rf_preds, target_names=["Real", "Spoof"]))

print("\nМетрики Модели №2 (Глубокая нейросеть PyTorch):")
print(classification_report(y_test_flat, nn_preds, target_names=["Real", "Spoof"]))

# Матрица ошибок лучшей модели (нейросети)
print("\nМатрица ошибок для нейросети:")
print(confusion_matrix(y_test_flat, nn_preds))

# Вывод победителя по качеству
best_model_name = "AntiSpoofNet (PyTorch)" if nn_f1 >= rf_f1 else "Random Forest"
print(f"\nЛучшая модель по метрике F1-Score: {best_model_name}")


In [ ]:
#5. Этап 5. Экспорт лучшей модели в ONNX и Инференс-пайплайн

logger.info("ЭТАП 5: ОПТИМИЗАЦИЯ И РАЗВЕРТЫВАНИЕ")

# Конвертируем победившую нейросеть в ONNX формат
onnx_path = "anti_spoof_net.onnx"
with torch.inference_mode():
    dummy_tensor = torch.randn(1, 3, 224, 224, device=DEVICE)
    torch.onnx.export(
        nn_model, dummy_tensor, onnx_path, export_params=True, opset_version=16,
        do_constant_folding=True, input_names=['input'], output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )
logger.info(f"Вычислительный граф ONNX успешно записан в: {onnx_path}")

# Построение высокопроизводительного класса-клиента
class AntiSpoofPredictor:
    def __init__(self, model_path: str) -> None:
        available_providers = ort.get_available_providers()
        if torch.cuda.is_available() and 'TensorRTExecutionProvider' in available_providers:
            logger.info("Обнаружен GPU. Компиляция TensorRT Engine (FP16)...")
            providers = [
                ('TensorRTExecutionProvider', {
                    'device_id': 0, 'trt_max_workspace_size': 2147483648, 'trt_fp16_enable': True
                }),
                ('CUDAExecutionProvider', {'device_id': 0})
            ]
        else:
            logger.warning("TensorRT недоступен. Переключение на CPU.")
            providers = ['CPUExecutionProvider']
            
        self.session = ort.InferenceSession(model_path, providers=providers)
        self.input_name = self.session.get_inputs().name
        self.labels = ["REAL (Живое лицо)", "SPOOF (Атака/Фейк)"]
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def predict(self, image_path: str) -> Tuple[str, float]:
        with Image.open(image_path).convert('RGB') as img:
            tensor_np = self.transform(img).unsqueeze(0).numpy()
        raw_outputs = self.session.run(None, {self.input_name: tensor_np})
        # Безопасный Softmax
        exp_logits = np.exp(raw_outputs - np.max(raw_outputs, axis=1, keepdims=True))
        probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
        class_idx = int(np.argmax(probabilities, axis=1))
        return self.labels[class_idx], float(probabilities[0, class_idx])

predictor = AntiSpoofPredictor(onnx_path)


In [ ]:
#6. Спид-тест (Бенчмарк) с сохранением результатов в CSV и автозагрузкой

def run_performance_benchmark(engine: AntiSpoofPredictor, iterations: int = 200, log_filename: str = "benchmark_results.csv") -> None:
    dummy_payload = np.random.randn(1, 3, 224, 224).astype(np.float32)
    
    # Прогрев
    for _ in range(30):
        _ = engine.session.run(None, {engine.input_name: dummy_payload})
        
    start_timestamp = time.time()
    for _ in range(iterations):
        _ = engine.session.run(None, {engine.input_name: dummy_payload})
    end_timestamp = time.time()
    
    total_time = end_timestamp - start_timestamp
    fps = iterations / total_time
    latency_ms = (total_time / iterations) * 1000
    
    active_providers = engine.session.get_providers()
    backend_name = "TensorRT" if "TensorRTExecutionProvider" in active_providers else "CPU"
    
    print("\n" + "="*50)
    print(f" ИТОГОВЫЕ МЕТРИКИ ДЛЯ ПРЕЗЕНТАЦИИ (ПРОЕКТ)")
    print(f"Выбранный бэкенд инференса: {backend_name}")
    print(f"Пропускная способность (FPS): {fps:.2f} кадров в секунду")
    print(f"Средняя задержка (Latency):  {latency_ms:.2f} мс")
    print("="*50 + "\n")
    
    file_exists = os.path.exists(log_filename)
    with open(log_filename, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["Временная метка", "Бэкенд", "FPS (кадры/сек)", "Задержка (мс)", "Полный список провайдеров"])
        writer.writerow([time.strftime("%Y-%m-%d %H:%M:%S"), backend_name, f"{fps:.2f}", f"{latency_ms:.2f}", str(active_providers)])
    logger.info(f"Метрики сохранены в: {log_filename}")

run_performance_benchmark(predictor)

# Автоматическое скачивание отчета результатов, если код запущен в Google Colab
try:
    from google.colab import files
    logger.info("Скачивание файла результатов на локальный компьютер...")
    files.download("benchmark_results.csv")
except ImportError:
    logger.info("Локальная среда исполнения. Автоматическая выгрузка через Google API пропущена.")
